# Stage 4.5 v2 -- Train TrackNetV2 from scratch

Trains TrackNetV2 from random initialization on user-labeled frames.
Replaces the v1 fine-tune approach (which failed; see KNOWN_ISSUES.md).

**Key differences from v1:**
- Random init, NOT loading Dettor's weights.
- MSE loss against Gaussian heatmap target (no weighted BCE, no pos_weight).
- Spatial augmentation (rotation +/-5deg, translation +/-10%) to defeat
  positional memorization.
- Multi-session resumable via per-epoch checkpoints.
- Adam lr=1e-3 with cosine annealing; weight decay 1e-5.

**Prerequisites in `MyDrive/pickleball/`:**
- `_tracknet_model.py` -- vendored from `stages/track_ball/`
- `training_data.zip` -- output of `prepare_training_data.py`, zipped
- `checkpoints/` -- folder (empty on first run; auto-populated after epoch 1)

**Runtime:** Runtime > Change runtime type > A100 (or best available).

**Outputs (back to Drive):**
- `tracknet_v2_trained_v<N>.pt` -- best-val-loss weights (saved at end)
- `train_log_v<N>.json` -- per-epoch loss + hyperparameters (updated each epoch)
- `checkpoints/latest.pt` -- full state for resume (optimizer + scheduler + epoch)
- `checkpoints/epoch_<N>.pt` -- per-epoch backup

Bump `VERSION` in Cell 2 if you start a NEW training run from scratch.
Re-run with the same VERSION to resume from the latest checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type.')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
import hashlib
from pathlib import Path

VERSION = 1  # bump for each NEW training run; keep same to resume

DRIVE_ROOT = Path('/content/drive/MyDrive/pickleball')
MODEL_PY     = DRIVE_ROOT / '_tracknet_model.py'
DATA_ZIP     = DRIVE_ROOT / 'training_data.zip'
CKPT_DIR     = DRIVE_ROOT / 'checkpoints'

WEIGHTS_OUT  = DRIVE_ROOT / f'tracknet_v2_trained_v{VERSION}.pt'
LOG_OUT      = DRIVE_ROOT / f'train_log_v{VERSION}.json'
CKPT_LATEST  = CKPT_DIR / 'latest.pt'

LOCAL_DATA = Path('/content/training')

for p in (MODEL_PY, DATA_ZIP):
    if not p.exists():
        raise FileNotFoundError(f'Missing required input: {p}')
CKPT_DIR.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print(f'_tracknet_model.py SHA-256: {sha256(MODEL_PY)}')
print(f'training_data.zip size: {DATA_ZIP.stat().st_size / 1e9:.2f} GB')
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Resume from: {CKPT_LATEST} (exists: {CKPT_LATEST.exists()})')
print(f'Output weights -> {WEIGHTS_OUT}')
print(f'Output log     -> {LOG_OUT}')


In [ ]:
import json
import time
import zipfile

if LOCAL_DATA.exists() and any(LOCAL_DATA.iterdir()):
    print(f'{LOCAL_DATA} already populated; skipping unzip')
else:
    t0 = time.time()
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(LOCAL_DATA)
    print(f'unzipped in {time.time() - t0:.1f}s')

metadata_path = None
for candidate in [
    LOCAL_DATA / 'metadata.json',
    LOCAL_DATA / 'training' / 'metadata.json',
    LOCAL_DATA / 'data' / 'training' / 'metadata.json',
]:
    if candidate.exists():
        LOCAL_DATA = candidate.parent
        metadata_path = candidate
        break
if metadata_path is None:
    for depth in ('*/metadata.json', '*/*/metadata.json'):
        hits = list(LOCAL_DATA.glob(depth))
        if len(hits) == 1:
            metadata_path = hits[0]
            LOCAL_DATA = metadata_path.parent
            break
if metadata_path is None:
    raise FileNotFoundError(f'metadata.json not found under {LOCAL_DATA}')

with open(metadata_path) as f:
    metadata = json.load(f)

n_train_files = len(list((LOCAL_DATA / 'train').glob('*.npz')))
n_val_files   = len(list((LOCAL_DATA / 'val').glob('*.npz')))
print(f'data dir:      {LOCAL_DATA}')
print(f'metadata says: train={metadata["n_train"]}, val={metadata["n_val"]}')
print(f'on disk:       train={n_train_files}, val={n_val_files}')
print(f'val source:    {metadata["val_source_video"]}')
print(f'heatmap sigma: {metadata["heatmap_sigma"]}')
assert n_train_files == metadata['n_train']
assert n_val_files == metadata['n_val']


## Initialize TrackNet from scratch

Imports `_tracknet_model.py` from Drive. Creates a fresh TrackNetV2 with
random weights -- NOT loading Dettor's weights this time (see v1 lessons
in KNOWN_ISSUES.md). Runs a forward pass on a zero tensor to verify
shape and output range.


In [ ]:
import sys

if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))
if '_tracknet_model' in sys.modules:
    del sys.modules['_tracknet_model']
from _tracknet_model import TrackNet  # noqa: E402

DEVICE = torch.device('cuda')

model = TrackNet(in_channels=9, out_channels=3).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'fresh TrackNet (random init); parameters: {n_params:,}')

model.eval()
with torch.no_grad():
    dummy = torch.zeros(1, 9, 288, 512, device=DEVICE)
    y = model(dummy)
print(f'forward pass: input {tuple(dummy.shape)} -> output {tuple(y.shape)}')
print(f'output range on zeros: [{y.min().item():.4f}, {y.max().item():.4f}]')
assert y.shape == (1, 3, 288, 512), f'unexpected output shape {y.shape}'
assert 0.0 <= y.min().item() and y.max().item() <= 1.0, \
    f'output not in [0,1]: min={y.min().item()} max={y.max().item()}'
print('sanity checks passed')


## Dataset with v2 augmentation

Loads `.npz` samples. Casts uint8 input to float32/255 and float16 target
to float32.

**v2 augmentations:**
- Horizontal flip (50%)
- Brightness +/-15%, contrast +/-15%, per-color RGB shift +/-5%
- Random rotation +/-5 degrees (spatial; consistent across input + target)
- Random translation +/-10% of frame dimensions (spatial; consistent)

Rotation + translation are the v2 additions. They explicitly defeat
positional memorization (v1 learned that pixel (993, 273) is "always a
ball" because trees were always there). These augmentations do NOT
change the camera-position assumption: the camera is still in a far
corner ~6ft high; the ball is still detected from the same viewpoint
distribution. They only break "this pixel = ball."


In [ ]:
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader

MODEL_H = 288
MODEL_W = 512
ROTATION_DEG_MAX = 5.0
TRANSLATE_FRAC_MAX = 0.10


def _apply_affine_input(x_9chw, rot_deg, tx, ty):
    """Apply same rotation + translation to all 9 channels (3 frames x RGB).
    x_9chw: (9, H, W) float32. tx, ty in PIXELS, NOT fraction."""
    import cv2
    c, h, w = x_9chw.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rot_deg, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty
    out = np.empty_like(x_9chw)
    for ch in range(c):
        out[ch] = cv2.warpAffine(
            x_9chw[ch], M, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101,
        )
    return out


def _apply_affine_target(y_3chw, rot_deg, tx, ty):
    """Apply same rotation + translation to all 3 target channels.
    Use the same border mode so the Gaussian shape is preserved cleanly."""
    import cv2
    c, h, w = y_3chw.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rot_deg, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty
    out = np.empty_like(y_3chw)
    for ch in range(c):
        out[ch] = cv2.warpAffine(
            y_3chw[ch], M, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0.0,
        )
    return out


class BallDataset(Dataset):
    def __init__(self, root, split, augment):
        self.root = Path(root) / split
        self.files = sorted(self.root.glob('*.npz'))
        if not self.files:
            raise FileNotFoundError(f'no .npz files in {self.root}')
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        x = d['input'].astype(np.float32) / 255.0  # (9, H, W)
        y = d['target'].astype(np.float32)         # (3, H, W)

        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                x = x[:, :, ::-1].copy()
                y = y[:, :, ::-1].copy()

            # Photometric: brightness/contrast/RGB shift
            b = 1.0 + (random.random() - 0.5) * 0.3
            x = np.clip(x * b, 0, 1)
            c = 1.0 + (random.random() - 0.5) * 0.3
            mean = x.mean()
            x = np.clip((x - mean) * c + mean, 0, 1)
            # Channel layout: [R0,G0,B0,R1,G1,B1,R2,G2,B2]
            for color in range(3):
                shift = (random.random() - 0.5) * 0.1
                x[color::3] = np.clip(x[color::3] + shift, 0, 1)

            # Spatial: rotation + translation, same affine for input + target
            rot_deg = (random.random() - 0.5) * 2 * ROTATION_DEG_MAX
            tx = (random.random() - 0.5) * 2 * TRANSLATE_FRAC_MAX * MODEL_W
            ty = (random.random() - 0.5) * 2 * TRANSLATE_FRAC_MAX * MODEL_H
            x = _apply_affine_input(x, rot_deg, tx, ty)
            y = _apply_affine_target(y, rot_deg, tx, ty)

        return torch.from_numpy(x), torch.from_numpy(y)


train_ds = BallDataset(LOCAL_DATA, 'train', augment=True)
val_ds   = BallDataset(LOCAL_DATA, 'val',   augment=False)
print(f'train: {len(train_ds)} samples (with spatial+photometric aug)')
print(f'val:   {len(val_ds)} samples (no aug)')


## Hyperparameters, optimizer, scheduler

Adam lr=1e-3 (higher than v1's 1e-4 because we're training from scratch).
Cosine annealing from 1e-3 to 1e-5 across the full epoch budget.
Weight decay 1e-5. MSE loss against the Gaussian target -- no
pos_weight, no BCE.


In [ ]:
BATCH_SIZE = 8       # A100 has 40GB; v1 used 4 on T4 16GB. bump.
LR = 1e-3
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-5
EPOCHS = 50
PATIENCE = 10
WALL_BUDGET_SECONDS = 11.5 * 3600  # Pro sessions up to ~12h; leave margin

optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                             weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN)

# MSE loss; sum-then-mean over all positions (default 'mean' reduction)
loss_fn = torch.nn.MSELoss()

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

steps_per_train_epoch = len(train_loader)
steps_per_val_epoch   = len(val_loader)
print(f'batch size:        {BATCH_SIZE}')
print(f'train steps/epoch: {steps_per_train_epoch}')
print(f'val   steps/epoch: {steps_per_val_epoch}')
print(f'epochs:            up to {EPOCHS}, patience: {PATIENCE}')
print(f'wall budget:       {WALL_BUDGET_SECONDS/3600:.2f}h')


## Resume from checkpoint (if present)

If `checkpoints/latest.pt` exists, load model + optimizer + scheduler
state and the epoch counter. Otherwise start from epoch 0.


In [ ]:
import copy

start_epoch = 0
best_val_loss = float('inf')
best_state = None
best_epoch = None
patience_counter = 0
log_entries = []

if CKPT_LATEST.exists():
    print(f'resuming from {CKPT_LATEST}')
    ckpt = torch.load(CKPT_LATEST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch']  # next epoch to run
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    best_epoch = ckpt.get('best_epoch', None)
    patience_counter = ckpt.get('patience_counter', 0)
    log_entries = ckpt.get('log_entries', [])
    if 'best_state' in ckpt and ckpt['best_state'] is not None:
        best_state = ckpt['best_state']
    print(f'  resumed at epoch {start_epoch} '
          f'(best val loss so far: {best_val_loss:.4f} at epoch {best_epoch})')
    print(f'  patience counter: {patience_counter}/{PATIENCE}')
    print(f'  prior log entries: {len(log_entries)}')
else:
    print('no checkpoint found; starting from epoch 0')


## Training loop with per-epoch checkpoints

Each epoch:
1. Train through all samples.
2. Validate.
3. Update LR scheduler.
4. Write `checkpoints/latest.pt` (full state) and `checkpoints/epoch_N.pt`.
5. Update train_log_v<N>.json on Drive.
6. If patience exhausted or wall budget exceeded, break.

If interrupted, just re-run the notebook in a new session -- the resume
cell above will pick up where we left off.


In [ ]:
import time
import datetime as dt


def write_checkpoint(epoch, best_val_loss, best_epoch, patience_counter,
                     log_entries, best_state, path):
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'patience_counter': patience_counter,
        'log_entries': log_entries,
        'best_state': best_state,
    }, path)


def write_log(log_entries, stopped_reason, path):
    payload = {
        'schema_version': 1,
        'version': VERSION,
        'weights_path': str(WEIGHTS_OUT),
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'epochs_trained': len(log_entries),
        'stopped_reason': stopped_reason,
        'hyperparameters': {
            'batch_size': BATCH_SIZE,
            'learning_rate_initial': LR,
            'learning_rate_min': LR_MIN,
            'weight_decay': WEIGHT_DECAY,
            'patience': PATIENCE,
            'max_epochs': EPOCHS,
            'wall_budget_seconds': WALL_BUDGET_SECONDS,
            'augmentation': ('hflip 50%, brightness +/-15%, contrast +/-15%, '
                             'RGB shift +/-5%, rotation +/-5deg, '
                             'translation +/-10%'),
            'optimizer': 'Adam',
            'loss': 'MSE',
            'scheduler': 'cosine annealing',
        },
        'dataset': {
            'n_train': metadata['n_train'],
            'n_val': metadata['n_val'],
            'heatmap_sigma': metadata['heatmap_sigma'],
            'val_source_video': metadata['val_source_video'],
            'train_source_videos': metadata['train_source_videos'],
        },
        'per_epoch': log_entries,
        'last_updated_utc': dt.datetime.now(dt.UTC).isoformat().replace(
            '+00:00', 'Z'),
    }
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2)


stopped_reason = None
training_start = time.time()

for epoch in range(start_epoch, EPOCHS):
    elapsed_before = time.time() - training_start
    if elapsed_before > WALL_BUDGET_SECONDS:
        stopped_reason = (f'wall_budget exceeded before epoch {epoch} '
                          f'({elapsed_before:.0f}s)')
        print(stopped_reason)
        break

    epoch_start = time.time()
    model.train()
    tloss_sum = 0.0
    n = 0
    for step, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        pred = model(x)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tloss_sum += loss.item() * x.size(0)
        n += x.size(0)
        if step % 200 == 0:
            print(f'  epoch {epoch+1} step {step}/{steps_per_train_epoch} '
                  f'loss={loss.item():.6f}')
    train_loss = tloss_sum / n

    model.eval()
    vloss_sum = 0.0
    n = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pred = model(x)
            loss = loss_fn(pred, y)
            vloss_sum += loss.item() * x.size(0)
            n += x.size(0)
    val_loss = vloss_sum / n

    scheduler.step()
    cur_lr = optimizer.param_groups[0]['lr']

    epoch_seconds = time.time() - epoch_start
    cumulative = time.time() - training_start

    entry = {
        'epoch': epoch + 1,  # 1-indexed in the log
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': cur_lr,
        'epoch_seconds': epoch_seconds,
        'cumulative_seconds': cumulative,
    }
    log_entries.append(entry)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1
        patience_counter = 0
        marker = ' (best)'
    else:
        patience_counter += 1
        marker = f' (no improvement, patience {patience_counter}/{PATIENCE})'

    print(f'epoch {epoch+1}: train={train_loss:.6f} val={val_loss:.6f} '
          f'lr={cur_lr:.6f} ({epoch_seconds:.0f}s){marker}')

    # Per-epoch checkpoint
    write_checkpoint(epoch + 1, best_val_loss, best_epoch, patience_counter,
                     log_entries, best_state, CKPT_LATEST)
    epoch_ckpt = CKPT_DIR / f'epoch_{epoch+1:03d}.pt'
    write_checkpoint(epoch + 1, best_val_loss, best_epoch, patience_counter,
                     log_entries, best_state, epoch_ckpt)
    write_log(log_entries, 'in_progress', LOG_OUT)

    if patience_counter >= PATIENCE:
        stopped_reason = f'early stopping at epoch {epoch+1}'
        print(stopped_reason)
        break

if stopped_reason is None:
    stopped_reason = f'completed all {EPOCHS} epochs'
    print(stopped_reason)

if best_state is not None:
    print(f'\nbest val loss: {best_val_loss:.6f} at epoch {best_epoch}')
else:
    print('\nWARNING: no best state captured this session')


## Save final outputs

Writes the best-val-loss weights as `tracknet_v2_trained_v<N>.pt` and
updates `train_log_v<N>.json` with the final stopped_reason. Per-epoch
checkpoints in `checkpoints/` are kept for resumability of subsequent
training runs.


In [ ]:
if best_state is not None:
    torch.save(best_state, WEIGHTS_OUT)
    print(f'wrote {WEIGHTS_OUT} '
          f'({WEIGHTS_OUT.stat().st_size / 1e6:.1f} MB)')
else:
    print('no best state to save (training did not produce a usable model)')

write_log(log_entries, stopped_reason, LOG_OUT)
print(f'wrote {LOG_OUT}')
print(f'\nstopped_reason: {stopped_reason}')
print(f'epochs trained this session + prior: {len(log_entries)}')


## Next steps

If training completed (or you got to a good val-loss plateau):

1. Download tracknet_v2_trained_v1.pt from Drive to local data/models/.
2. Run Stage 4.5 validation locally (see stages/finetune_ball_model/
   validate.py for the exact command; outdoor video is the held-out
   validation source). Acceptance: detection_rate_at_10px >= 0.80.
3. If validation passes, re-run Stage 4 smoke test on test_clip
   with the new --weights path.
4. If validation fails, run tools/diag_heatmaps.py to identify the
   failure mode before training v3.

If training was interrupted before completing: just re-run this
notebook in a new Colab session. The resume cell will pick up
from the latest checkpoint.